# DD Startup Results Comparison Tool

This notebook provides a flexible interface to compare results across multiple HDF5 analysis files.

## Features
- Load and compare any number of analysis files
- Multiple comparison modes:
  - **Best N**: Top N combinations by chosen metric
  - **Worst N**: Bottom N combinations by chosen metric
  - **Quartile Analysis**: Statistics for top/bottom quartiles
  - **Random Sample**: Random selection for distribution analysis
- Excel export with separate sheets per file
- Summary statistics across all files

## Quick Start
1. Configure your files and analysis mode in Section 1
2. Run all cells
3. Find your Excel comparison in the `outputs/` directory

## Section 1: Configuration

Configure your analysis parameters here.

### 🆕 Modular Approach

This allows you to extract:
- Best values from each quartile
- Median values from each quartile  
- Top N best/worst overall
- Specific percentiles
- And more!

**Example use case:** "I want the best value and median from each quartile"
```python
ANALYSIS_STRATEGIES = [
    {'name': 'Q1_Best', 'type': 'quartile_best', 'quartile': 'q1'},
    {'name': 'Q1_Median', 'type': 'quartile_median', 'quartile': 'q1'},
    {'name': 'Q2_Best', 'type': 'quartile_best', 'quartile': 'q2'},
    {'name': 'Q2_Median', 'type': 'quartile_median', 'quartile': 'q2'},
    # ... etc
]
```

The notebook will process all strategies for each file and create comprehensive Excel sheets for comparison.


In [7]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Files to analyze (folders or .h5 files)
# If empty, will use the latest file found in outputs/
FILES_TO_ANALYZE = [
    '20251109_113840_parametric_lump',  # Folder name (relative to outputs/)
    '20251108_125645_parametric_T_seeded',  # Folder name
    # 'path/to/specific/file.h5',  # Or specific file path
]

# Metrics to use for comparison (can be a single string or a list)
# Single metric: COMPARISON_METRICS = 'unrealized_profits'
# Multiple metrics: COMPARISON_METRICS = ['unrealized_profits', 't_startup', 'Q_DD']
# Common options: 'unrealized_profits', 't_startup', 'Q_DD', 'E_lost', 'TBR_DDn', etc.
COMPARISON_METRICS = ['unrealized_profits', 't_startup']

# ============================================================================
# ANALYSIS STRATEGIES
# ============================================================================
# Define multiple extraction strategies to run on each file.
# Each strategy is a dict with 'name', 'type', and type-specific parameters.
# 
# Available types:
#   - 'best_n': Extract top N best (lowest) values
#   - 'worst_n': Extract top N worst (highest) values
#   - 'quartile_best': Extract best value from specified quartile
#   - 'quartile_median': Extract median value from specified quartile
#   - 'quartile_worst': Extract worst value from specified quartile
#   - 'quartile_sample': Extract N random samples from specified quartile
#   - 'percentile': Extract value at specific percentile
#   - 'random': Extract N random samples from entire dataset

ANALYSIS_STRATEGIES = [
    # Best overall
    #{'name': 'Top_10_Best', 'type': 'best_n', 'n': 10},
    
    # Best and Median from each quartile
    {'name': 'Q1_Best', 'type': 'quartile_best', 'quartile': 'q1'},
    {'name': 'Q1_Median', 'type': 'quartile_median', 'quartile': 'q1'},
    {'name': 'Q2_Best', 'type': 'quartile_best', 'quartile': 'q2'},
    {'name': 'Q2_Median', 'type': 'quartile_median', 'quartile': 'q2'},
    {'name': 'Q3_Best', 'type': 'quartile_best', 'quartile': 'q3'},
    {'name': 'Q3_Median', 'type': 'quartile_median', 'quartile': 'q3'},
    {'name': 'Q4_Best', 'type': 'quartile_best', 'quartile': 'q4'},
    {'name': 'Q4_Median', 'type': 'quartile_median', 'quartile': 'q4'},
    
    # Optional: Add more strategies as needed
    # {'name': 'Top_5_Worst', 'type': 'worst_n', 'n': 5},
    # {'name': 'P95', 'type': 'percentile', 'percentile': 95},
]

# ============================================================================
# OUTPUT CONFIGURATION
# ============================================================================

# Display options
SHOW_ALL_OUTPUTS = True  # Show all output variables in console
SHOW_ALL_INPUTS = True   # Show all input parameters in console
MAX_DISPLAY = 3          # Max combinations to display per strategy in console

# Failed simulations export
EXPORT_FAILED_SIMS = True  # Set to False to skip exporting failed simulations
MAX_FAILED_TO_EXPORT = 10000  # Maximum number of failed sims to export per file

# ============================================================================
# PROCESS CONFIGURATION
# ============================================================================

# Convert COMPARISON_METRICS to list if single string
if isinstance(COMPARISON_METRICS, str):
    COMPARISON_METRICS = [COMPARISON_METRICS]

# Generate file labels for Excel naming
file_labels = []
for file_path in FILES_TO_ANALYZE:
    # Extract meaningful name from path
    if '/' in file_path or '\\' in file_path:
        name = file_path.split('/')[-1].split('\\')[-1]
    else:
        name = file_path
    
    # Clean up name (remove timestamps, common prefixes)
    # Example: '20251109_113840_parametric_lump' -> 'lump'
    if 'parametric' in name.lower():
        parts = name.lower().split('parametric')
        if len(parts) > 1:
            label = parts[1].strip('_').replace('_', '')
        else:
            label = name.replace('parametric', '').strip('_').replace('_', '')
    else:
        label = name.split('_')[-1] if '_' in name else name
    
    file_labels.append(label)

print("✅ Configuration loaded")
print(f"   Strategies: {len(ANALYSIS_STRATEGIES)} defined")
print(f"   Metrics: {', '.join(COMPARISON_METRICS)}")
print(f"   Strategy examples: {', '.join([s['name'] for s in ANALYSIS_STRATEGIES[:3]])}{'...' if len(ANALYSIS_STRATEGIES) > 3 else ''}")
print(f"   Files: {len(FILES_TO_ANALYZE) if FILES_TO_ANALYZE else 'Using latest'}")
if len(COMPARISON_METRICS) > 1:
    print(f"   Output: {len(COMPARISON_METRICS)} Excel files will be created (one per metric)")


✅ Configuration loaded
   Strategies: 8 defined
   Metrics: unrealized_profits, t_startup
   Strategy examples: Q1_Best, Q1_Median, Q2_Best...
   Files: 2
   Output: 2 Excel files will be created (one per metric)


## Section 2: Setup and Imports

Import required libraries and helper functions.

In [8]:
# ============================================================================
# IMPORTS
# ============================================================================

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add parent directory to path for imports
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from ddstartup.postprocessing.postprocess_functions import (
    find_latest_h5_file,
    get_input_parameters
)

# Check for openpyxl
import openpyxl

print("✅ Imports complete")

✅ Imports complete


## Section 3: Helper Functions

Define functions for data loading and processing.

In [9]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_h5_efficiently(h5_file, target_metric):
    """
    Load HDF5 file efficiently - only loads the target metric initially,
    then loads full data only for selected rows.
    
    Returns:
        tuple: (full_df, metric_series, total_rows)
    """
    import h5py
    import hdf5plugin
    
    with h5py.File(h5_file, 'r') as f:
        # Get total rows from file attributes
        total_rows = f.attrs.get('total_combinations', None)
        if total_rows is None:
            # Try alternative attribute name
            total_rows = f.attrs.get('n_samples', None)
        if total_rows is None:
            raise ValueError(f"Could not find total row count in {h5_file.name}")
        
        # Check if metric exists (datasets are at top level)
        if target_metric not in f.keys():
            available = list(f.keys())
            raise ValueError(f"Metric '{target_metric}' not found in {h5_file.name}. Available: {available[:10]}")
        
        # Load only the target metric for initial filtering
        metric_data = f[target_metric][:]
    
    return None, metric_data, total_rows


def load_selected_rows(h5_file, indices):
    """
    Load only specific rows from HDF5 file.
    Only loads 1-D datasets (scalars) - skips time-series data.
    
    Args:
        h5_file: Path to HDF5 file
        indices: Array of row indices to load
    
    Returns:
        DataFrame with selected rows
    """
    import h5py
    import hdf5plugin
    
    data = {}
    with h5py.File(h5_file, 'r') as f:
        # Load all 1-D datasets (skip time-series data)
        for key in f.keys():
            if isinstance(f[key], h5py.Dataset):
                # Only load 1-D datasets (scalars per row)
                if len(f[key].shape) == 1:
                    data[key] = f[key][:][indices]
                # Skip multi-dimensional datasets (time-series, etc.)
    
    return pd.DataFrame(data)


def apply_strategy(metric_data, strategy):
    """
    Apply an analysis strategy to extract specific indices.
    
    Args:
        metric_data: Array of metric values
        strategy: Dictionary with 'name', 'type', and type-specific parameters
    
    Returns:
        tuple: (indices, description)
    """
    strategy_type = strategy['type']
    total_rows = len(metric_data)
    
    if strategy_type == 'best_n':
        n = strategy['n']
        indices = np.argpartition(metric_data, min(n, total_rows-1))[:n]
        indices = indices[np.argsort(metric_data[indices])]
        desc = f"Best {n} (lowest values)"
        
    elif strategy_type == 'worst_n':
        n = strategy['n']
        indices = np.argpartition(metric_data, max(0, total_rows-n))[-n:]
        indices = indices[np.argsort(metric_data[indices])[::-1]]
        desc = f"Worst {n} (highest values)"
        
    elif strategy_type == 'quartile_best':
        quartile = strategy['quartile']
        q_indices = get_quartile_indices(metric_data, quartile)
        best_in_q = np.argmin(metric_data[q_indices])
        indices = np.array([q_indices[best_in_q]])
        desc = f"Best value in {quartile.upper()}"
        
    elif strategy_type == 'quartile_median':
        quartile = strategy['quartile']
        q_indices = get_quartile_indices(metric_data, quartile)
        sorted_q_values = np.sort(metric_data[q_indices])
        median_idx = len(sorted_q_values) // 2
        median_val = sorted_q_values[median_idx]
        # Find the actual index of this median value
        q_sorted_indices = q_indices[np.argsort(metric_data[q_indices])]
        indices = np.array([q_sorted_indices[median_idx]])
        desc = f"Median value in {quartile.upper()}"
        
    elif strategy_type == 'quartile_worst':
        quartile = strategy['quartile']
        q_indices = get_quartile_indices(metric_data, quartile)
        worst_in_q = np.argmax(metric_data[q_indices])
        indices = np.array([q_indices[worst_in_q]])
        desc = f"Worst value in {quartile.upper()}"
        
    elif strategy_type == 'quartile_sample':
        quartile = strategy['quartile']
        n = strategy.get('n', 10)
        q_indices = get_quartile_indices(metric_data, quartile)
        if len(q_indices) > n:
            np.random.seed(42)
            sampled = np.random.choice(q_indices, size=n, replace=False)
            indices = sampled[np.argsort(metric_data[sampled])]
        else:
            indices = q_indices[np.argsort(metric_data[q_indices])]
        desc = f"Sample of {len(indices)} from {quartile.upper()}"
        
    elif strategy_type == 'percentile':
        percentile = strategy['percentile']
        p_value = np.percentile(metric_data, percentile)
        # Find closest value to percentile
        idx = np.argmin(np.abs(metric_data - p_value))
        indices = np.array([idx])
        desc = f"Value at {percentile}th percentile"
        
    elif strategy_type == 'random':
        n = strategy['n']
        np.random.seed(42)
        indices = np.random.choice(total_rows, size=min(n, total_rows), replace=False)
        indices = indices[np.argsort(metric_data[indices])]
        desc = f"Random sample of {len(indices)}"
        
    else:
        raise ValueError(f"Unknown strategy type: {strategy_type}")
    
    return indices, desc


def get_quartile_indices(metric_data, quartile):
    """
    Get indices for a specific quartile.
    
    Args:
        metric_data: Array of metric values
        quartile: 'q1', 'q2', 'q3', or 'q4'
    
    Returns:
        Array of indices in the specified quartile
    """
    total_rows = len(metric_data)
    sorted_indices = np.argsort(metric_data)
    
    q_ranges = {
        'q1': (0, total_rows // 4),
        'q2': (total_rows // 4, total_rows // 2),
        'q3': (total_rows // 2, 3 * total_rows // 4),
        'q4': (3 * total_rows // 4, total_rows)
    }
    
    start, end = q_ranges[quartile]
    return sorted_indices[start:end]


def format_value_for_display(value, param_name=''):
    """
    Format a value for console display.
    """
    if param_name == 't_startup':
        years = value / (365.25 * 24 * 3600)
        return f"{years:.4g} years ({value:.4g} s)"
    elif isinstance(value, (int, np.integer)):
        return f"{value:,}"
    elif isinstance(value, (float, np.floating)):
        return f"{value:.4g}"
    else:
        return str(value)


print("✅ Helper functions defined")


✅ Helper functions defined


## Section 4: File Collection

Collect and validate HDF5 files to analyze.

In [10]:
# ============================================================================
# COLLECT FILES
# ============================================================================

outputs_dir = Path.cwd()  # We're already in outputs/
h5_files = []

if FILES_TO_ANALYZE:
    print(f"Collecting files from configuration...")
    for file_path in FILES_TO_ANALYZE:
        p = Path(file_path)
        
        # Handle relative paths (relative to outputs/)
        if not p.is_absolute():
            p = outputs_dir / p
        
        if p.is_dir():
            # Find H5 files in directory
            h5_in_dir = list(p.glob('*.h5'))
            if h5_in_dir:
                h5_files.extend(h5_in_dir)
                print(f"  ✅ Found {len(h5_in_dir)} file(s) in {p.name}")
            else:
                print(f"  ⚠️  No .h5 files in {p.name}")
        elif p.suffix == '.h5' and p.exists():
            h5_files.append(p)
            print(f"  ✅ Added {p.name}")
        else:
            print(f"  ⚠️  Invalid or missing: {file_path}")
else:
    # Use latest file
    print("No files specified, searching for latest...")
    latest = find_latest_h5_file(outputs_dir)
    if latest:
        h5_files = [latest]
        print(f"  ✅ Using latest: {latest.name}")
    else:
        print("  ❌ No HDF5 files found in outputs/")

if not h5_files:
    raise ValueError("No HDF5 files to analyze. Check your FILES_TO_ANALYZE configuration.")

print(f"\n{'='*80}")
print(f"📂 Ready to analyze {len(h5_files)} file(s)")
print(f"{'='*80}")

  ✅ Found 1 file(s) in 20251109_113840_parametric_lump
  ✅ Found 1 file(s) in 20251108_125645_parametric_T_seeded

📂 Ready to analyze 2 file(s)


## Section 5: Analysis and Comparison

Process each file and extract comparison data.

In [11]:
# ============================================================================
# PROCESS FILES - LOOP OVER METRICS
# ============================================================================

# Store results for all metrics: {metric_name: {file_name: {strategy_name: df}}}
all_metrics_results = {}
all_metrics_statistics = {}
all_metrics_metadata = {}

for COMPARISON_METRIC in COMPARISON_METRICS:
    
    print(f"\n{'#'*80}")
    print(f"# PROCESSING METRIC: {COMPARISON_METRIC}")
    print(f"{'#'*80}")
    
    all_results = {}  # Store DataFrames: {file_name: {strategy_name: df}}
    all_statistics = {}  # Store statistics: {file_name: {strategy_name: stats}}
    file_metadata = {}  # Store metadata
    
    for h5_file in h5_files:
        file_name = h5_file.stem
        
        print(f"\n{'='*80}")
        print(f"📁 Processing: {h5_file.name}")
        print(f"{'='*80}")
        
        try:
            # Load metric data AND success flag
            print(f"Loading metric '{COMPARISON_METRIC}' and filtering successful runs...")
            
            import h5py
            import hdf5plugin
            
            with h5py.File(h5_file, 'r') as f:
                total_rows = f.attrs.get('total_combinations', f.attrs.get('n_samples', None))
                
                # Check if metric exists
                if COMPARISON_METRIC not in f.keys():
                    available = list(f.keys())
                    raise ValueError(f"Metric '{COMPARISON_METRIC}' not found. Available: {available[:10]}")
                
                # Load metric and success flag
                metric_data = f[COMPARISON_METRIC][:]
                
                # Load sol_success if available
                if 'sol_success' in f.keys():
                    success_flag = f['sol_success'][:]
                    # Filter to only successful runs
                    valid_mask = success_flag.astype(bool)
                    valid_indices = np.where(valid_mask)[0]
                    
                    if len(valid_indices) == 0:
                        print(f"  ⚠️  No successful runs found in this file!")
                        continue
                    
                    print(f"  Found {total_rows:,} total combinations")
                    print(f"  Filtered to {len(valid_indices):,} successful runs ({100*len(valid_indices)/total_rows:.1f}%)")
                    
                    # Create filtered metric data (only successful runs)
                    filtered_metric_data = metric_data[valid_indices]
                else:
                    # No success flag, use all data
                    print(f"  Found {total_rows:,} total combinations (no sol_success flag)")
                    valid_indices = np.arange(total_rows)
                    filtered_metric_data = metric_data
            
            # Initialize storage for this file
            all_results[file_name] = {}
            all_statistics[file_name] = {}
            file_metadata[file_name] = {
                'total_rows': total_rows,
                'successful_rows': len(valid_indices)
            }
            
            # Use configured strategies
            strategies = ANALYSIS_STRATEGIES
            print(f"  Running {len(strategies)} analysis strategies...")
            
            # Process each strategy
            for strategy in strategies:
                strategy_name = strategy['name']
                
                try:
                    print(f"\n  🔍 Strategy: {strategy_name}")
                    
                    # Get indices based on strategy (applied to filtered data)
                    filtered_indices, mode_desc = apply_strategy(filtered_metric_data, strategy)
                    print(f"     Selection: {mode_desc} ({len(filtered_indices)} rows)")
                    
                    # Map back to original indices
                    original_indices = valid_indices[filtered_indices]
                    
                    # Load only selected rows
                    df = load_selected_rows(h5_file, original_indices)
                    
                    # Verify we got successful runs
                    if 'sol_success' in df.columns:
                        n_failed = (~df['sol_success'].astype(bool)).sum()
                        if n_failed > 0:
                            print(f"     ⚠️  WARNING: {n_failed} failed runs in selection (should be 0!)")
                    
                    # Add metadata columns
                    df.insert(0, 'Strategy', strategy_name)
                    df.insert(1, 'Rank_in_Strategy', range(1, len(df) + 1))
                    df.insert(2, 'Source_File', file_name)
                    
                    # Get input/output parameters (only once per file)
                    if 'input_params' not in file_metadata[file_name]:
                        input_params = get_input_parameters(df, COMPARISON_METRIC, filename=str(h5_file))
                        output_params = [col for col in df.columns if col not in input_params + ['Strategy', 'Rank_in_Strategy', 'Source_File']]
                        file_metadata[file_name]['input_params'] = input_params
                        file_metadata[file_name]['output_params'] = output_params
                    else:
                        input_params = file_metadata[file_name]['input_params']
                        output_params = file_metadata[file_name]['output_params']
                    
                    # Store results
                    all_results[file_name][strategy_name] = df
                    
                    # Calculate statistics
                    stats = {
                        'min': df[COMPARISON_METRIC].min(),
                        'max': df[COMPARISON_METRIC].max(),
                        'mean': df[COMPARISON_METRIC].mean(),
                        'median': df[COMPARISON_METRIC].median(),
                        'std': df[COMPARISON_METRIC].std() if len(df) > 1 else 0.0,
                        'count': len(df),
                        'description': mode_desc
                    }
                    all_statistics[file_name][strategy_name] = stats
                    
                    # Display preview (only first few strategies to avoid clutter)
                    strategy_idx = strategies.index(strategy)
                    if strategy_idx < 3:  # Show first 3 strategies
                        print(f"     Stats: min={stats['min']:.6g}, max={stats['max']:.6g}, mean={stats['mean']:.6g}")
                        
                        n_display = min(MAX_DISPLAY, len(df))
                        for idx in range(n_display):
                            row = df.iloc[idx]
                            print(f"       #{idx+1}: {COMPARISON_METRIC}={row[COMPARISON_METRIC]:.6g}")
                    elif strategy_idx == 3:
                        print(f"     ... (showing first 3 strategies only, {len(strategies)-3} more processed)")
                    
                except Exception as e:
                    print(f"     ❌ Error with strategy '{strategy_name}': {e}")
                    import traceback
                    traceback.print_exc()
                    continue
            
            print(f"\n✅ File processed successfully ({len(all_results[file_name])} strategies)")
            
        except Exception as e:
            print(f"❌ Error processing {h5_file.name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Store results for this metric
    all_metrics_results[COMPARISON_METRIC] = all_results
    all_metrics_statistics[COMPARISON_METRIC] = all_statistics
    all_metrics_metadata[COMPARISON_METRIC] = file_metadata
    
    print(f"\n{'='*80}")
    print(f"✅ Metric '{COMPARISON_METRIC}' processed successfully")
    print(f"   Files: {len(all_results)}")
    if not USE_LEGACY_MODE and len(all_results) > 0:
        total_strategies = sum(len(strategies) for strategies in all_results.values())
        print(f"   Total strategy results: {total_strategies}")
    print(f"{'='*80}")

print(f"\n{'#'*80}")
print(f"✅ ALL METRICS PROCESSED SUCCESSFULLY")
print(f"   Metrics: {len(COMPARISON_METRICS)}")
print(f"   Files per metric: {len(h5_files)}")
print(f"{'#'*80}")



################################################################################
# PROCESSING METRIC: unrealized_profits
################################################################################

📁 Processing: ddstartup_20251109_113840_parametric_lump.h5
Loading metric 'unrealized_profits' and filtering successful runs...
  Found 2,025,000 total combinations
  Filtered to 1,788,768 successful runs (88.3%)
  Running 8 analysis strategies...

  🔍 Strategy: Q1_Best
  Found 2,025,000 total combinations
  Filtered to 1,788,768 successful runs (88.3%)
  Running 8 analysis strategies...

  🔍 Strategy: Q1_Best


     Selection: Best value in Q1 (1 rows)
     Stats: min=2.96052e+06, max=2.96052e+06, mean=2.96052e+06
       #1: unrealized_profits=2.96052e+06

  🔍 Strategy: Q1_Median
     Stats: min=2.96052e+06, max=2.96052e+06, mean=2.96052e+06
       #1: unrealized_profits=2.96052e+06

  🔍 Strategy: Q1_Median
     Selection: Median value in Q1 (1 rows)
     Selection: Median value in Q1 (1 rows)
     Stats: min=4.68811e+07, max=4.68811e+07, mean=4.68811e+07
       #1: unrealized_profits=4.68811e+07

  🔍 Strategy: Q2_Best
     Selection: Best value in Q2 (1 rows)
     Stats: min=4.68811e+07, max=4.68811e+07, mean=4.68811e+07
       #1: unrealized_profits=4.68811e+07

  🔍 Strategy: Q2_Best
     Selection: Best value in Q2 (1 rows)
     Stats: min=1.32121e+08, max=1.32121e+08, mean=1.32121e+08
       #1: unrealized_profits=1.32121e+08

  🔍 Strategy: Q2_Median
     Stats: min=1.32121e+08, max=1.32121e+08, mean=1.32121e+08
       #1: unrealized_profits=1.32121e+08

  🔍 Strategy: Q2_Median
     Selec

NameError: name 'USE_LEGACY_MODE' is not defined

## Section 6: Excel Export

Create comprehensive Excel comparison file.

In [ ]:
# ============================================================================
# CREATE EXCEL COMPARISON FILES (ONE PER METRIC)
# ============================================================================

if not all_metrics_results:
    print("❌ No results to export")
else:
    print(f"\n{'='*80}")
    print("📊 Creating Excel comparison files...")
    print(f"{'='*80}\n")
    
    excel_files_created = []
    
    # Loop over each metric
    for COMPARISON_METRIC in COMPARISON_METRICS:
        
        all_results = all_metrics_results[COMPARISON_METRIC]
        all_statistics = all_metrics_statistics[COMPARISON_METRIC]
        file_metadata = all_metrics_metadata[COMPARISON_METRIC]
        
        if not all_results:
            print(f"⚠️  No results for metric '{COMPARISON_METRIC}', skipping...")
            continue
        
        print(f"\n📋 Creating Excel for metric: {COMPARISON_METRIC}")
        print("-" * 80)
        
        # Generate Excel filename
        # Format: compare_file1_file2_metric.xlsx
        file_parts = '_'.join(file_labels[:3])  # Limit to first 3 files
        if len(file_labels) > 3:
            file_parts += f"_plus{len(file_labels)-3}more"
        
        excel_filename = f"compare_{file_parts}_{COMPARISON_METRIC}.xlsx"
        excel_path = outputs_dir / excel_filename
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            
            # Sheet 1: Summary Statistics
            print(f"  Creating Summary sheet...")
            summary_data = []
            for file_name, strategies in all_results.items():
                for strategy_name, df_result in strategies.items():
                    stats = all_statistics[file_name][strategy_name]
                    meta = file_metadata[file_name]
                    summary_data.append({
                        'File': file_name,
                        'Strategy': strategy_name,
                        'Description': stats['description'],
                        'Total_Combinations': meta['total_rows'],
                        'N_Selected': stats['count'],
                        f'{COMPARISON_METRIC}_Min': stats['min'],
                        f'{COMPARISON_METRIC}_Max': stats['max'],
                        f'{COMPARISON_METRIC}_Mean': stats['mean'],
                        f'{COMPARISON_METRIC}_Median': stats['median'],
                        f'{COMPARISON_METRIC}_Std': stats['std'],
                    })
            
            df_summary = pd.DataFrame(summary_data)
            df_summary.to_excel(writer, sheet_name='Summary', index=False)
            print(f"    ✅ Summary sheet created")
            
            # Sheet 2: All Results Combined
            print(f"  Creating All_Results sheet...")
            all_dfs = []
            for file_name, strategies in all_results.items():
                for strategy_name, df_result in strategies.items():
                    all_dfs.append(df_result)
            
            if all_dfs:
                df_all = pd.concat(all_dfs, ignore_index=True)
                # Sort by metric for easy comparison
                df_all = df_all.sort_values(COMPARISON_METRIC)
                df_all.insert(0, 'Global_Rank', range(1, len(df_all) + 1))
                df_all.to_excel(writer, sheet_name='All_Results', index=False)
                print(f"    ✅ All_Results sheet created")
            
            # Sheets 3-N: Per-File sheets (all strategies for each file)
            print(f"  Creating per-file sheets...")
            for file_name, strategies in all_results.items():
                # Combine all strategies for this file
                file_dfs = []
                for strategy_name, df_result in strategies.items():
                    file_dfs.append(df_result)
                
                if file_dfs:
                    df_file = pd.concat(file_dfs, ignore_index=True)
                    df_file = df_file.sort_values([COMPARISON_METRIC])
                    
                    # Truncate sheet name if needed (Excel limit: 31 chars)
                    sheet_name = file_name[:31] if len(file_name) > 31 else file_name
                    df_file.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"    ✅ Sheet '{sheet_name}' created ({len(strategies)} strategies)")
            
            # Additional sheets: Per-Strategy sheets (all files for each strategy)
            if not USE_LEGACY_MODE and len(all_results) > 1:
                print(f"  Creating per-strategy sheets...")
                # Get all unique strategy names
                all_strategy_names = set()
                for strategies in all_results.values():
                    all_strategy_names.update(strategies.keys())
                
                for strategy_name in sorted(all_strategy_names):
                    strategy_dfs = []
                    for file_name, strategies in all_results.items():
                        if strategy_name in strategies:
                            strategy_dfs.append(strategies[strategy_name])
                    
                    if strategy_dfs:
                        df_strategy = pd.concat(strategy_dfs, ignore_index=True)
                        df_strategy = df_strategy.sort_values(COMPARISON_METRIC)
                        
                        # Truncate sheet name (prefix with 'S_' for strategy)
                        sheet_name = f"S_{strategy_name}"[:31]
                        df_strategy.to_excel(writer, sheet_name=sheet_name, index=False)
                        print(f"    ✅ Sheet '{sheet_name}' created")
        
        excel_files_created.append(excel_filename)
        print(f"\n  ✅ Excel file created: {excel_filename}")
    
    print(f"\n{'='*80}")
    print(f"✅ ALL EXCEL FILES CREATED")
    print(f"{'='*80}")
    print(f"\n📁 Files created ({len(excel_files_created)}):")
    for filename in excel_files_created:
        print(f"   - {filename}")
    
    print(f"\n📋 Each file includes:")
    print(f"   - Summary: Statistics for all files and strategies")
    print(f"   - All_Results: All data combined and ranked by metric")
    print(f"   - Per-file sheets: All strategies for each file")
    if not USE_LEGACY_MODE and len(all_results) > 1:
        print(f"   - Per-strategy sheets: Cross-file comparison for each strategy")
    print(f"\n💡 Tip: Use filters and pivot tables in Excel for deeper analysis!")



📊 Creating Excel comparison files...


📋 Creating Excel for metric: unrealized_profits
--------------------------------------------------------------------------------
  Creating Summary sheet...
    ✅ Summary sheet created
  Creating All_Results sheet...
    ✅ All_Results sheet created
  Creating per-file sheets...
    ✅ Sheet 'ddstartup_20251109_113840_param' created (8 strategies)
    ✅ Sheet 'ddstartup_20251108_125645_param' created (8 strategies)
  Creating per-strategy sheets...
    ✅ Sheet 'S_Q1_Best' created
    ✅ Sheet 'S_Q1_Median' created
    ✅ Sheet 'S_Q2_Best' created
    ✅ Sheet 'S_Q2_Median' created
    ✅ Sheet 'S_Q3_Best' created
    ✅ Sheet 'S_Q3_Median' created
    ✅ Sheet 'S_Q4_Best' created
    ✅ Sheet 'S_Q4_Median' created

  ✅ Excel file created: compare_lump_tseeded_unrealized_profits.xlsx

📋 Creating Excel for metric: t_startup
--------------------------------------------------------------------------------
  Creating Summary sheet...
    ✅ Summary sheet create

## Section 6b: Export Failed Simulations (Optional)

Export input parameters that led to failed simulations for debugging and analysis.

In [ ]:
# ============================================================================
# EXPORT FAILED SIMULATIONS
# ============================================================================

if EXPORT_FAILED_SIMS:
    print(f"\n{'='*80}")
    print("🔍 EXPORTING FAILED SIMULATIONS")
    print(f"{'='*80}\n")
    
    failed_files_created = []
    
    for h5_file in h5_files:
        file_name = h5_file.stem
        
        print(f"Processing: {h5_file.name}")
        
        try:
            import h5py
            import hdf5plugin
            
            with h5py.File(h5_file, 'r') as f:
                # Check if sol_success exists
                if 'sol_success' not in f.keys():
                    print(f"  ⚠️  No 'sol_success' flag found, skipping...")
                    continue
                
                # Load success flag
                success_flag = f['sol_success'][:]
                failed_mask = ~success_flag.astype(bool)
                failed_indices = np.where(failed_mask)[0]
                
                total_rows = len(success_flag)
                n_failed = len(failed_indices)
                
                if n_failed == 0:
                    print(f"  ✅ No failed simulations (100% success rate)")
                    continue
                
                print(f"  Found {n_failed:,} failed simulations ({100*n_failed/total_rows:.1f}%)")
                
                # Limit number of failed sims to export
                if n_failed > MAX_FAILED_TO_EXPORT:
                    print(f"  ⚠️  Limiting export to {MAX_FAILED_TO_EXPORT:,} failed simulations (random sample)")
                    np.random.seed(42)
                    failed_indices = np.random.choice(failed_indices, size=MAX_FAILED_TO_EXPORT, replace=False)
                    n_failed = len(failed_indices)
                
                # Load failed rows
                print(f"  Loading {n_failed:,} failed simulation rows...")
                df_failed = load_selected_rows(h5_file, failed_indices)
                
                # Identify input vs output parameters
                # Known outputs (excludes inputs)
                known_outputs = {
                    # Simulation results
                    'E_lost', 'Q_DD', 'Q_DT_eq', 'P_DT_eq',
                    't_startup', 'unrealized_profits',
                    # Metadata
                    'error', 'sol_success', 'linear_index',
                    # Time-dependent outputs (vectors)
                    'N_ifc', 'N_ofc', 'N_st', 'N_T', 'N_D', 'N_He3_plasma',
                    'P_DDn', 'P_DDp', 'P_DT', 'P_fusion', 'P_alpha',
                    'TBE', 'time'
                }
                
                # Get all input columns (those not in known outputs)
                all_input_params = [col for col in df_failed.columns if col not in known_outputs]
                
                # Add error and sol_success for debugging
                export_columns = all_input_params + ['error', 'sol_success']
                
                # Create DataFrame with all input parameters + error message
                df_export = df_failed[export_columns].copy()
                
                # Add metadata
                df_export.insert(0, 'Failed_Index', failed_indices)
                df_export.insert(1, 'Source_File', file_name)
                
                # Decode error messages if they're bytes
                if 'error' in df_export.columns:
                    df_export['error'] = df_export['error'].apply(
                        lambda x: x.decode('utf-8') if isinstance(x, bytes) else str(x)
                    )
                
                # Display which input parameters are being exported
                print(f"  Exporting {len(all_input_params)} input parameters:")
                print(f"    {', '.join(all_input_params)}")
                
                # Generate Excel filename
                file_label = file_labels[h5_files.index(h5_file)] if h5_files.index(h5_file) < len(file_labels) else file_name
                excel_filename = f"failed_simulations_{file_label}.xlsx"
                excel_path = outputs_dir / excel_filename
                
                # Export to Excel
                print(f"  Creating Excel file...")
                with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                    
                    # Sheet 1: Failed simulations data
                    df_export.to_excel(writer, sheet_name='Failed_Simulations', index=False)
                    
                    # Sheet 2: Summary by error type
                    if 'error' in df_export.columns:
                        error_summary = df_export.groupby('error').size().reset_index(name='Count')
                        error_summary = error_summary.sort_values('Count', ascending=False)
                        error_summary['Percentage'] = 100 * error_summary['Count'] / len(df_export)
                        error_summary.to_excel(writer, sheet_name='Error_Summary', index=False)
                        
                        print(f"  📊 Error types found:")
                        for _, row in error_summary.head(5).iterrows():
                            print(f"     - {row['error']}: {row['Count']:,} ({row['Percentage']:.1f}%)")
                        if len(error_summary) > 5:
                            print(f"     ... ({len(error_summary)-5} more error types)")
                
                failed_files_created.append(excel_filename)
                print(f"  ✅ Exported to: {excel_filename}\n")
                
        except Exception as e:
            print(f"  ❌ Error exporting failed simulations: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"{'='*80}")
    print(f"✅ FAILED SIMULATIONS EXPORT COMPLETE")
    print(f"{'='*80}")
    
    if failed_files_created:
        print(f"\n📁 Files created ({len(failed_files_created)}):")
        for filename in failed_files_created:
            print(f"   - {filename}")
        print(f"\n💡 Use these files to:")
        print(f"   - Debug why certain parameter combinations failed")
        print(f"   - Identify problematic parameter ranges")
        print(f"   - Adjust constraints or solver settings")
    else:
        print(f"\n✅ No failed simulations found in any files!")

else:
    print(f"\n{'='*80}")
    print("ℹ️  Failed simulations export disabled (set EXPORT_FAILED_SIMS = True to enable)")
    print(f"{'='*80}")



🔍 EXPORTING FAILED SIMULATIONS

Processing: ddstartup_20251109_113840_parametric_lump.h5
  Found 236,232 failed simulations (11.7%)
  ⚠️  Limiting export to 10,000 failed simulations (random sample)
  Loading 10,000 failed simulation rows...
  Exporting 16 input parameters:
    I_target, P_aux, P_aux_DT_eq, TBR_DDn, TBR_DT, T_i, V_plasma, capacity_factor, eta_th, n_He3, n_tot, price_of_electricity, tau_ifc, tau_ofc, tau_p_He3, tau_p_T
  Creating Excel file...
  Exporting 16 input parameters:
    I_target, P_aux, P_aux_DT_eq, TBR_DDn, TBR_DT, T_i, V_plasma, capacity_factor, eta_th, n_He3, n_tot, price_of_electricity, tau_ifc, tau_ofc, tau_p_He3, tau_p_T
  Creating Excel file...
  📊 Error types found:
     - Physics solver failed or t_startup infinite: 10,000 (100.0%)
  📊 Error types found:
     - Physics solver failed or t_startup infinite: 10,000 (100.0%)
  ✅ Exported to: failed_simulations_lump.xlsx

Processing: ddstartup_20251108_125645_parametric_T_seeded.h5
  Found 197,028 failed 

## Section 7: Comparison Analysis (Optional)

Additional analysis and visualization of the comparison results.

In [ ]:
# ============================================================================
# ADDITIONAL ANALYSIS (FOR ALL METRICS)
# ============================================================================

print(f"\n{'='*80}")
print("📈 COMPARISON ANALYSIS")
print(f"{'='*80}\n")

if not all_metrics_results:
    print("❌ No results to analyze")
else:
    # Loop over each metric
    for COMPARISON_METRIC in COMPARISON_METRICS:
        
        all_results = all_metrics_results.get(COMPARISON_METRIC, {})
        all_statistics = all_metrics_statistics.get(COMPARISON_METRIC, {})
        
        if not all_results:
            continue
        
        print(f"\n{'─'*80}")
        print(f"📊 METRIC: {COMPARISON_METRIC}")
        print(f"{'─'*80}\n")
        
        if len(all_results) == 1:
            # Single file analysis
            file_name = list(all_results.keys())[0]
            strategies = all_results[file_name]
            
            print(f"Single file analysis: {file_name}\n")
            print(f"Comparing {len(strategies)} strategies by {COMPARISON_METRIC}:\n")
            
            # Find best and worst strategies
            strategy_means = {name: all_statistics[file_name][name]['mean'] for name in strategies.keys()}
            best_strategy = min(strategy_means, key=strategy_means.get)
            worst_strategy = max(strategy_means, key=strategy_means.get)
            
            print(f"🏆 Best performing strategy (lowest mean {COMPARISON_METRIC}):")
            print(f"   {best_strategy}")
            print(f"   Mean: {strategy_means[best_strategy]:.6g}")
            print(f"   Min:  {all_statistics[file_name][best_strategy]['min']:.6g}")
            
            print(f"\n📉 Worst performing strategy (highest mean {COMPARISON_METRIC}):")
            print(f"   {worst_strategy}")
            print(f"   Mean: {strategy_means[worst_strategy]:.6g}")
            print(f"   Max:  {all_statistics[file_name][worst_strategy]['max']:.6g}")
            
            # Rank all strategies
            print(f"\n📊 Strategy rankings by mean {COMPARISON_METRIC}:")
            ranked_strategies = sorted(strategy_means.items(), key=lambda x: x[1])
            for rank, (name, mean) in enumerate(ranked_strategies[:5], 1):  # Show top 5
                desc = all_statistics[file_name][name]['description']
                print(f"   {rank}. {name}: {mean:.6g} ({desc})")
            if len(ranked_strategies) > 5:
                print(f"   ... ({len(ranked_strategies)-5} more strategies)")

        else:
            # Multi-file comparison
            print(f"Comparing {len(all_results)} files by {COMPARISON_METRIC}:\n")
            
            # Calculate overall best value per file (across all strategies)
            file_best_values = {}
            file_best_strategies = {}
            for file_name, strategies in all_results.items():
                best_val = float('inf')
                best_strat = None
                for strategy_name in strategies.keys():
                    val = all_statistics[file_name][strategy_name]['min']
                    if val < best_val:
                        best_val = val
                        best_strat = strategy_name
                file_best_values[file_name] = best_val
                file_best_strategies[file_name] = best_strat
            
            # Find best and worst performing files
            best_file = min(file_best_values, key=file_best_values.get)
            worst_file = max(file_best_values, key=file_best_values.get)
            
            print(f"🏆 Best performing file (lowest {COMPARISON_METRIC} overall):")
            print(f"   {best_file}")
            print(f"   Best value: {file_best_values[best_file]:.6g}")
            print(f"   Found in strategy: {file_best_strategies[best_file]}")
            
            print(f"\n📉 Worst performing file (highest {COMPARISON_METRIC} overall):")
            print(f"   {worst_file}")
            print(f"   Best value: {file_best_values[worst_file]:.6g}")
            print(f"   Found in strategy: {file_best_strategies[worst_file]}")
            
            # Calculate improvement
            if file_best_values[worst_file] != 0:
                improvement = ((file_best_values[worst_file] - file_best_values[best_file]) / file_best_values[worst_file]) * 100
                print(f"\n💡 Performance difference: {improvement:.2f}% improvement from worst to best")
            
            # Rank all files
            print(f"\n📊 File rankings by best {COMPARISON_METRIC}:")
            ranked_files = sorted(file_best_values.items(), key=lambda x: x[1])
            for rank, (name, best_val) in enumerate(ranked_files, 1):
                strategy = file_best_strategies[name]
                print(f"   {rank}. {name}: {best_val:.6g} (via {strategy})")
            
            # If using modular mode, show per-strategy comparison (only for first metric to avoid clutter)
            if not USE_LEGACY_MODE and COMPARISON_METRIC == COMPARISON_METRICS[0]:
                print(f"\n🔍 Per-Strategy Comparison (showing for {COMPARISON_METRIC} only):")
                
                # Get all unique strategy names
                all_strategy_names = set()
                for strategies in all_results.values():
                    all_strategy_names.update(strategies.keys())
                
                # Show only first 3 strategies to avoid clutter
                for idx, strategy_name in enumerate(sorted(all_strategy_names)):
                    if idx >= 3:
                        print(f"\n   ... ({len(all_strategy_names)-3} more strategies, see Excel for details)")
                        break
                    
                    print(f"\n   Strategy: {strategy_name}")
                    strategy_results = {}
                    for file_name, strategies in all_results.items():
                        if strategy_name in strategies:
                            val = all_statistics[file_name][strategy_name]['min']
                            strategy_results[file_name] = val
                    
                    if strategy_results:
                        best_file_for_strategy = min(strategy_results, key=strategy_results.get)
                        print(f"      Best file: {best_file_for_strategy} ({strategy_results[best_file_for_strategy]:.6g})")
                        
                        # Show all files for this strategy
                        for file_name in sorted(strategy_results, key=lambda x: strategy_results[x]):
                            print(f"         {file_name}: {strategy_results[file_name]:.6g}")

print(f"\n{'='*80}")
print("✅ ANALYSIS COMPLETE")
print(f"{'='*80}")



📈 COMPARISON ANALYSIS


────────────────────────────────────────────────────────────────────────────────
📊 METRIC: unrealized_profits
────────────────────────────────────────────────────────────────────────────────

Comparing 2 files by unrealized_profits:

🏆 Best performing file (lowest unrealized_profits overall):
   ddstartup_20251108_125645_parametric_T_seeded
   Best value: 248718
   Found in strategy: Q1_Best

📉 Worst performing file (highest unrealized_profits overall):
   ddstartup_20251109_113840_parametric_lump
   Best value: 2.96052e+06
   Found in strategy: Q1_Best

💡 Performance difference: 91.60% improvement from worst to best

📊 File rankings by best unrealized_profits:
   1. ddstartup_20251108_125645_parametric_T_seeded: 248718 (via Q1_Best)
   2. ddstartup_20251109_113840_parametric_lump: 2.96052e+06 (via Q1_Best)

🔍 Per-Strategy Comparison (showing for unrealized_profits only):

   Strategy: Q1_Best
      Best file: ddstartup_20251108_125645_parametric_T_seeded (2487